In [2]:
import pandas as pd
import numpy as np
import re
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

In [3]:
data = pd.read_csv('/Users/drishtant/Documents/Masters/CA/core market data/Project_details_1.3.csv')

In [4]:
# Suppose your dataset has two main columns: 'description' (project descriptions) and 'SDGs' (aligned SDGs)
# The 'SDGs' column contains lists of SDGs each project aligns with, e.g., "[1,3,5]"
# You may need to preprocess this column to work with your model effectively

In [5]:
# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["description"], padding="max_length", truncation=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [7]:
# Prepare the dataset
df = data
df['SDGs'] = df['SDGs'].apply(eval)  # Convert string representation of list to actual list
mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(df['SDGs'])


KeyError: 'SDGs'

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(df['description'], labels, test_size=0.1)
train_encodings = tokenizer(list(X_train), truncation=True, padding=True)
val_encodings = tokenizer(list(X_val), truncation=True, padding=True)

# Convert to torch dataset
class SDGDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)



In [ ]:
train_dataset = SDGDataset(train_encodings, y_train)
val_dataset = SDGDataset(val_encodings, y_val)

In [ ]:
# Training
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(mlb.classes_))

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()